In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load dataset
df = pd.read_csv('/home/ezreals_/Code/NoteBook/MachineLearning/Week5/Dataset/insurance.csv')
df.head()

In [ ]:
print("Shape dataset:", df.shape)
print()
df.info()

In [ ]:
df.describe()

In [ ]:
print("Jumlah nilai kosong per kolom:")
print(df.isnull().sum())

In [ ]:
# Distribusi target variable (charges)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['charges'], bins=50, color='steelblue', edgecolor='white')
plt.title('Distribusi Biaya Medis (charges)')
plt.xlabel('charges (USD)')
plt.ylabel('Frekuensi')

plt.subplot(1, 2, 2)
plt.hist(np.log(df['charges']), bins=50, color='darkorange', edgecolor='white')
plt.title('Distribusi Log(charges)')
plt.xlabel('log(charges)')
plt.ylabel('Frekuensi')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot fitur numerik vs charges
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
num_features = ['age', 'bmi', 'children']

for i, feat in enumerate(num_features):
    axes[i].scatter(df[feat], df['charges'], alpha=0.4, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('charges')
    axes[i].set_title(f'{feat} vs charges')

plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata charges per variabel kategorikal
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cat_features = ['smoker', 'sex', 'region']

for i, feat in enumerate(cat_features):
    df.groupby(feat)['charges'].mean().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Rata-rata charges per {feat}')
    axes[i].set_ylabel('Rata-rata charges (USD)')
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Korelasi fitur numerik
corr_matrix = df[['age', 'bmi', 'children', 'charges']].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriks Korelasi Fitur Numerik')
plt.tight_layout()
plt.show()

In [ ]:
# Encoding variabel kategorikal
df_encoded = df.copy()

le = LabelEncoder()
df_encoded['sex']    = le.fit_transform(df_encoded['sex'])     # female=0, male=1
df_encoded['smoker'] = le.fit_transform(df_encoded['smoker'])  # no=0, yes=1

# One-Hot Encoding untuk region
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

print("Kolom setelah encoding:", df_encoded.columns.tolist())
df_encoded.head()

In [ ]:
# Variabel bebas dan target
X = df_encoded.drop(columns=['charges'])
y = df_encoded['charges']

# Bagi 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Jumlah data asli  : {df.shape[0]}")
print(f"Jumlah data train : {X_train.shape[0]}")
print(f"Jumlah data test  : {X_test.shape[0]}")

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print(f"Intercept: {lr_model.intercept_:.2f}")
print("\nKoefisien per fitur:")
for feat, coef in zip(X.columns, lr_model.coef_):
    print(f"  {feat:30s}: {coef:.4f}")

In [ ]:
y_pred_lr = lr_model.predict(X_test)

r2_lr  = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print("=== Evaluasi Multiple Linear Regression ===")
print(f"R-squared (R²) : {r2_lr:.4f}")
print(f"MSE            : {mse_lr:,.2f}")
print(f"MAE            : {mae_lr:,.2f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Garis Ideal')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title('Multiple Linear Regression – Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature Scaling (wajib untuk SVR)
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train_scaled = sc_X.fit_transform(X_train)
X_test_scaled  = sc_X.transform(X_test)
y_train_scaled = sc_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

print("Feature scaling selesai.")

In [ ]:
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1, gamma=0.1)
svr_model.fit(X_train_scaled, y_train_scaled)

print(f"Kernel  : {svr_model.kernel}")
print(f"C       : {svr_model.C}")
print(f"Epsilon : {svr_model.epsilon}")
print(f"Gamma   : {svr_model.gamma}")

In [ ]:
y_pred_svr_scaled = svr_model.predict(X_test_scaled)
y_pred_svr = sc_y.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()

r2_svr  = r2_score(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

print("=== Evaluasi SVR (RBF, C=100, epsilon=0.1, gamma=0.1) ===")
print(f"R-squared (R²) : {r2_svr:.4f}")
print(f"MSE            : {mse_svr:,.2f}")
print(f"MAE            : {mae_svr:,.2f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_svr, alpha=0.5, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'b--', lw=2, label='Garis Ideal')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title('SVR (RBF) – Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
configs = [
    {'kernel': 'rbf',    'C': 1,    'epsilon': 0.1,  'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 10,   'epsilon': 0.1,  'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 100,  'epsilon': 0.1,  'gamma': 0.1},
    {'kernel': 'rbf',    'C': 1000, 'epsilon': 0.01, 'gamma': 0.01},
    {'kernel': 'linear', 'C': 10,   'epsilon': 0.1,  'gamma': 'scale'},
]

results = []
for cfg in configs:
    model = SVR(**cfg)
    model.fit(X_train_scaled, y_train_scaled)
    y_pred_inv = sc_y.inverse_transform(
        model.predict(X_test_scaled).reshape(-1, 1)
    ).ravel()
    results.append({
        'kernel' : cfg['kernel'],
        'C'      : cfg['C'],
        'epsilon': cfg['epsilon'],
        'gamma'  : str(cfg['gamma']),
        'R²'     : round(r2_score(y_test, y_pred_inv), 4),
        'MSE'    : round(mean_squared_error(y_test, y_pred_inv), 2),
        'MAE'    : round(mean_absolute_error(y_test, y_pred_inv), 2),
    })

results_df = pd.DataFrame(results)
print("Hasil Hyperparameter Tuning SVR:")
results_df

In [ ]:
best_svr = results_df.loc[results_df['R²'].idxmax()]

summary = pd.DataFrame({
    'Model': ['Multiple Linear Regression',
              f"SVR (kernel={best_svr['kernel']}, C={best_svr['C']})"],
    'R²'   : [round(r2_lr, 4), best_svr['R²']],
    'MSE'  : [round(mse_lr, 2), best_svr['MSE']],
    'MAE'  : [round(mae_lr, 2), best_svr['MAE']],
})
print("=== Perbandingan Model ===")
summary

In [ ]:
plt.figure(figsize=(8, 4))
colors = ['steelblue', 'darkorange']
bars = plt.bar(summary['Model'], summary['R²'], color=colors, edgecolor='white')
for bar, val in zip(bars, summary['R²']):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{val:.4f}", ha='center', va='bottom', fontsize=11)
plt.ylim(0, 1)
plt.ylabel('R² Score')
plt.title('Perbandingan R² – MLR vs SVR')
plt.tight_layout()
plt.show()

In [ ]:
print('''
========================================================
 ANALISIS HASIL
========================================================

[1] EXPLORATORY DATA ANALYSIS (EDA)
------------------------------------
- Distribusi 'charges' sangat right-skewed (miring kanan),
  artinya sebagian besar orang membayar biaya medis yang
  relatif rendah, namun ada sejumlah kecil individu dengan
  biaya sangat tinggi.

- Dari bar chart, perokok (smoker=yes) memiliki rata-rata
  biaya medis jauh lebih tinggi dibanding non-perokok.
  Ini adalah faktor paling dominan.

- Dari scatter plot, 'age' dan 'bmi' menunjukkan korelasi
  positif terhadap 'charges', meskipun polanya tidak murni
  linear – terlihat ada beberapa kelompok/cluster data.

[2] MULTIPLE LINEAR REGRESSION
--------------------------------
- Model ini berasumsi hubungan LINIER antara semua fitur
  dan target (charges).

- Koefisien terbesar dimiliki oleh 'smoker', membuktikan
  bahwa status merokok adalah prediktor terkuat biaya medis.

- Kelemahan: tidak dapat menangkap pola non-linear atau
  interaksi antar fitur (misal: kombinasi smoker + bmi tinggi
  menghasilkan lonjakan biaya yang tidak bisa dijelaskan
  secara linear).

- Kelebihan: mudah diinterpretasikan, cepat dilatih.

[3] SUPPORT VECTOR REGRESSION (SVR)
-------------------------------------
- SVR dengan kernel RBF mampu menangkap pola NON-LINEAR
  dalam data, sehingga lebih cocok untuk dataset ini
  yang memiliki cluster dalam distribusi charges.

- Feature scaling (StandardScaler) wajib dilakukan sebelum
  melatih SVR karena model sensitif terhadap skala fitur.
  Prediksi hasil scaling di-inverse-transform kembali ke
  skala asli (USD).

- Dari eksperimen hyperparameter tuning:
    * C (regularisasi): nilai C besar -> model lebih fit
      pada training data, namun risiko overfitting.
    * epsilon: margin toleransi error; nilai kecil berarti
      model lebih ketat dalam fitting.
    * gamma (kernel RBF): mengontrol jangkauan pengaruh
      satu data point; nilai kecil -> pengaruh luas.

[4] PERBANDINGAN MODEL
-----------------------
  Model                         R2      MSE            MAE
  Multiple Linear Regression    0.7836  33,596,915     4,181
  SVR (RBF, C=100, gamma=0.1)   0.8474  23,687,626     2,618

- SVR menghasilkan R² lebih tinggi (~0.85 vs ~0.78) dan
  error lebih rendah (MSE & MAE) dibanding MLR.

- Hal ini menunjukkan bahwa hubungan antara fitur dan
  biaya medis bersifat NON-LINEAR, sehingga SVR lebih
  mampu memodelkan pola tersebut.

[5] KESIMPULAN
---------------
- Untuk dataset Medical Cost ini, SVR dengan kernel RBF
  lebih unggul dari Multiple Linear Regression.

- Faktor terpenting yang mempengaruhi besarnya biaya medis
  adalah status merokok (smoker), diikuti oleh usia (age)
  dan BMI.

- Untuk improvement lebih lanjut, dapat dicoba:
    * GridSearchCV untuk tuning SVR secara otomatis
    * Transformasi logaritmik pada target (log-charges)
    * Model ensemble seperti Random Forest atau Gradient
      Boosting untuk performa yang lebih baik.
========================================================
''')